# 🎨 face2sketch — Phase 2: Finetune → Caricature (Kaggle)

**Load Phase 1 G weights, finetune on TwitterPicasso (184 pairs) for caricature style.**

### Prerequisites
- **GPU:** T4 x2 — enable in Notebook → Accelerator → GPU
- **Internet:** Enable in Notebook → Internet → On
- **Dataset:** Add `face2sketch` dataset as input (contains data.zip + pix2pix_best.pt)

### Strategy (v2)
- **λ_L1=200** — L1 does the heavy lifting (D dies early on 184 pairs)
- **50-epoch L1-only warmup** — build caricature structure first
- **LR=1e-4**, **DataParallel + Grad Accum** — leverage T4 x2

In [ ]:
# @title 1. Clone Repo + Install
import os, sys, shutil
BASE = '/kaggle/working/face2sketch'
os.makedirs(BASE, exist_ok=True)
os.chdir('/kaggle/working')

!git clone https://github.com/weseegod/face2sketch.git {BASE} 2>/dev/null || true
os.chdir(BASE)
!git pull origin main

!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"BASE: {BASE}")
print('\n✅  Setup complete.')

# Stick to BASE for all later cells
os.environ['FACE2SKETCH_BASE'] = BASE

In [ ]:
# @title 2. Copy Data + Checkpoint from Kaggle Dataset
import glob, os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

# Find the kaggle_dataset folder (contains dataset/ finetune/ test/)
data_root = None
ckpt_src = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dataset' in dirs and 'finetune' in dirs and 'test' in dirs:
        data_root = root
    if 'pix2pix_best.pt' in files:
        ckpt_src = os.path.join(root, 'pix2pix_best.pt')

# ── Copy data ──
if data_root:
    print(f'📂  Found data at: {data_root}')
    os.makedirs(f'{BASE}/data', exist_ok=True)
    for sub in ['dataset', 'finetune', 'test']:
        src = os.path.join(data_root, sub)
        dst = f'{BASE}/data/{sub}'
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
        n = len(os.listdir(os.path.join(dst, 'photos')))
        print(f'    data/{sub}/  →  {n} photos')
else:
    print('❌  dataset/finetune/test/ folders not found')

# ── Copy checkpoint ──
os.makedirs(f'{BASE}/checkpoints', exist_ok=True)
if ckpt_src:
    shutil.copy(ckpt_src, f'{BASE}/checkpoints/pix2pix_best.pt')
    print(f'📂  pix2pix_best.pt ✅')
else:
    print(f'❌  pix2pix_best.pt not found')

# ── Verify ──
def count(d): return len(os.listdir(d))//2 if os.path.isdir(d) else 0
ft = count(f'{BASE}/data/finetune/photos')
ds = count(f'{BASE}/data/dataset/photos')
ck = os.path.exists(f'{BASE}/checkpoints/pix2pix_best.pt')
print(f'\n✅  dataset={ds} pairs  |  finetune={ft} pairs  |  ckpt={"YES" if ck else "NO"}')

if ft == 0 or not ck:
    print('⛔  Cannot proceed.')
    raise SystemExit(1)

In [ ]:
# @title 3. Finetune — 100 epochs (~30 min T4 x2)
import os

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

assert torch.cuda.is_available(), "⚠️  Enable GPU in Notebook settings"
assert os.path.exists('checkpoints/pix2pix_best.pt'), "❌ Missing checkpoint — re-run cell 2"
assert os.path.isdir('data/finetune'), "❌ Missing finetune data — re-run cell 2"

print("🎯  Finetuning Phase 1 G on TwitterPicasso caricatures")
print("    λ_L1=200 | 50-epoch L1 warmup | LR=1e-4 | DataParallel + Grad Accum")
print(f"    Dataset: 184 pairs  |  Epochs: 100\n")

!python src/train.py --mode finetune \
    --config configs/pix2pix_phase2.yaml \
    --finetune checkpoints/pix2pix_best.pt \
    --device cuda \
    --name phase2_

print("\n✅  Finetuning complete → checkpoints/phase2_best.pt")

In [ ]:
# @title 4. Evaluate — Phase 2 Gate Check
import os, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

ckpt = 'checkpoints/phase2_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase2_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No Phase 2 checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data found.')
else:
    !python src/evaluate.py --checkpoint {ckpt} --mode quick --device cuda

    print('\n📊  Comparing Phase 1 vs Phase 2 outputs...')
    p1_ckpt = 'checkpoints/pix2pix_best.pt'
    if os.path.exists(p1_ckpt):
        !python src/evaluate.py --checkpoint {p1_ckpt} \
            --checkpoint2 {ckpt} --mode compare --device cuda
    print('   ✅  Comparison: outputs/phase1_vs_phase2.png')

In [ ]:
# @title 5. Save Results to Kaggle Output
import os, shutil, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
OUT  = '/kaggle/working'

# ====================== ZIP CHECKPOINTS ONLY ======================
import zipfile

checkpoint_dir = f'{BASE}/checkpoints'
zip_path = f'{OUT}/checkpoints.zip'

if os.path.exists(checkpoint_dir):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(checkpoint_dir):
            for file in files:
                file_path = os.path.join(root, file)
                # Keep the folder structure inside the zip (checkpoints/best.pt, etc.)
                arcname = os.path.relpath(file_path, OUT)
                zipf.write(file_path, arcname)
    
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f'✅ checkpoints.zip created ({size_mb:.1f} MB)')
    print('📥 Just download **checkpoints.zip** from the Kaggle Output tab')
    print('   → It contains only: checkpoints/best.pt + checkpoints/final.pt')
else:
    print('⚠️ checkpoints/ folder not found — nothing to zip')